# 02 — Text Preprocessing

This notebook demonstrates the text preprocessing pipeline.

**Objectives:**
- Apply and visualize each preprocessing step
- Justify the cleaning decisions
- Compare text before and after cleaning
- Prepare the dataset for TF-IDF vectorization

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np

from src.data_loader import load_and_merge_data, remove_duplicates
from src.preprocessing import (
    clean_text, combine_title_text,
    remove_html_tags, remove_urls, remove_email_addresses,
    remove_special_characters, normalize_whitespace, remove_reuters_marker
)
print("Libraries loaded.")

## 1. Load and Merge Data

In [ ]:
df = load_and_merge_data()
print(f"Dataset loaded: {df.shape[0]} articles, {df.shape[1]} columns")

## 2. Handle Duplicates

Duplicates are removed before further processing to prevent data leakage.

In [ ]:
df, n_removed = remove_duplicates(df, subset=["title", "text"])
print(f"Removed {n_removed} duplicate articles")
print(f"Remaining: {len(df)} articles")

## 3. Demonstrate Each Preprocessing Step

In [ ]:
sample_real = df[df["label"] == 1].iloc[0]
sample_text = sample_real["text"]
print("ORIGINAL TEXT (first 500 chars):")
print(sample_text[:500])
print(f"\nLength: {len(sample_text)} characters")

In [ ]:
step1 = remove_html_tags(sample_text)
print("After HTML removal (first 500 chars):")
print(step1[:500])

In [ ]:
step2 = remove_urls(step1)
print("After URL removal (first 500 chars):")
print(step2[:500])

In [ ]:
step3 = remove_email_addresses(step2)
print("After email removal (first 500 chars):")
print(step3[:500])

In [ ]:
step4 = remove_reuters_marker(step3)
print("After Reuters marker removal (first 500 chars):")
print(step4[:500])

In [ ]:
step5 = remove_special_characters(step4)
print("After special character removal (first 500 chars):")
print(step5[:500])

In [ ]:
step6 = normalize_whitespace(step5)
print("After whitespace normalization (first 500 chars):")
print(step6[:500])

In [ ]:
full_clean = clean_text(sample_text, remove_source_markers=True)
print("FULLY CLEANED (first 500 chars):")
print(full_clean[:500])
print(f"\nOriginal: {len(sample_text)} chars -> Cleaned: {len(full_clean)} chars")

## 4. Apply to Entire Dataset

In [ ]:
df["combined_text"] = df.apply(
    lambda row: combine_title_text(row.get("title", ""), row.get("text", "")),
    axis=1
)
df["clean_text"] = df["combined_text"].apply(
    lambda x: clean_text(x, remove_source_markers=True)
)

empty_mask = df["clean_text"].str.strip() == ""
n_empty = empty_mask.sum()
print(f"Articles with empty text after cleaning: {n_empty}")
if n_empty > 0:
    df = df[~empty_mask].reset_index(drop=True)
    print(f"Removed {n_empty} empty articles. Remaining: {len(df)}")
print(f"\nDataset ready for vectorization: {len(df)} articles")

## 5. Before/After Comparison

In [ ]:
for lbl in [0, 1]:
    name = "Fake" if lbl == 0 else "Real"
    idx = df[df["label"] == lbl].index[0]
    print(f"\n{'='*60}")
    print(f"Example {name} Article")
    print(f"ORIGINAL: {df.loc[idx, 'combined_text'][:200]}...")
    print(f"CLEANED:  {df.loc[idx, 'clean_text'][:200]}...")

## 6. Summary of Preprocessing Decisions

| Step | Action | Justification |
|------|--------|---------------|
| HTML tags | Removed | Noise |
| URLs | Removed | Not useful for classification |
| Email addresses | Removed | Not useful |
| Reuters markers | Removed | Reduces source-based bias |
| Special characters | Removed (keeping apostrophes, hyphens) | Noise reduction |
| Whitespace | Normalized | Consistent tokenization |
| Lowercase | Delegated to TfidfVectorizer | Separation of concerns |
| Stop words | Delegated to TfidfVectorizer | Consistency |
| Numbers | Preserved | Can carry meaning |
| Negations | Preserved | Important semantic content |